# High-Performance DataFrames with Polars

## The Dataset: BTS On-Time Performance

**1. Download the 2023 monthly files**

In [44]:
import urllib.request
import os

os.makedirs("../data/flights", exist_ok=True)

base_url = "https://transtats.bts.gov/PREZIP/"
filename_template = (
    "On_Time_Reporting_Carrier_On_Time_Performance"
    "_1987_present_2023_{month}.zip"
)

for month in range(1, 13):
    filename = filename_template.format(month=month)
    dest = f"../data/flights/{filename}"
    if not os.path.exists(dest):
        print(f"Downloading month {month:02d}…")
        urllib.request.urlretrieve(base_url + filename, dest)
    else:
        print(f"Already exists: {dest}")

print("All files downloaded.")

All files downloaded.


**2. Extract and verify**

In [45]:
import zipfile
import glob
import os

zip_files = sorted(glob.glob(
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance"
    "_1987_present_2023_*.zip"
))

print(f"Found {len(zip_files)} zip files. Starting extraction...\n")

for zf in zip_files:
    try:
        with zipfile.ZipFile(zf) as z:
            z.extractall("../data/flights/")
            print(f"✅ Extracted: {os.path.basename(zf)}")
    except zipfile.BadZipFile:
        print(f"❌ SKIPPED - Not a valid zip: {os.path.basename(zf)}  ← Delete and re-download this one")
    except Exception as e:
        print(f"❌ Error with {os.path.basename(zf)}: {e}")

print("\nExtraction complete.")

Found 12 zip files. Starting extraction...

✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_1.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_10.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_11.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_12.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_2.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_3.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_4.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_5.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_6.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_7.zip
✅ Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_8.zip
✅ Extracted: On_Time_Reporting_Carrier_O

**3. Load December locally**

In [46]:
import polars as pl

jan_path = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv"
)

df = pl.read_csv(jan_path, infer_schema_length=10000)

print(df.shape)    # (570394, 110)
print(df.schema)

(570394, 110)
Schema({'Year': Int64, 'Quarter': Int64, 'Month': Int64, 'DayofMonth': Int64, 'DayOfWeek': Int64, 'FlightDate': String, 'Reporting_Airline': String, 'DOT_ID_Reporting_Airline': Int64, 'IATA_CODE_Reporting_Airline': String, 'Tail_Number': String, 'Flight_Number_Reporting_Airline': Int64, 'OriginAirportID': Int64, 'OriginAirportSeqID': Int64, 'OriginCityMarketID': Int64, 'Origin': String, 'OriginCityName': String, 'OriginState': String, 'OriginStateFips': Int64, 'OriginStateName': String, 'OriginWac': Int64, 'DestAirportID': Int64, 'DestAirportSeqID': Int64, 'DestCityMarketID': Int64, 'Dest': String, 'DestCityName': String, 'DestState': String, 'DestStateFips': Int64, 'DestStateName': String, 'DestWac': Int64, 'CRSDepTime': Int64, 'DepTime': String, 'DepDelay': Float64, 'DepDelayMinutes': Float64, 'DepDel15': Float64, 'DepartureDelayGroups': Int64, 'DepTimeBlk': String, 'TaxiOut': Float64, 'WheelsOff': String, 'WheelsOn': String, 'TaxiIn': Float64, 'CRSArrTime': Int64, 'Arr

## Why Polars

### What makes Polars fast

In [47]:
import polars as pl
import pandas as pd
import glob
import time

cols = ["Reporting_Airline", "ArrDelay", "DepDelay",
        "Distance", "Cancelled"]

pattern = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_*.csv"
)

# Load full year into memory once for a fair comparison
df_pd = pd.concat(
    [pd.read_csv(f, usecols=cols) for f in sorted(glob.glob(pattern))]
)
df_pl = pl.from_pandas(df_pd)

print(f"Rows: {len(df_pd):,}")   # ~7,000,000

# --- Pandas ---
t0 = time.perf_counter()
for _ in range(50):
    df_pd.groupby("Reporting_Airline")["ArrDelay"].agg(["mean", "std"])
pandas_time = time.perf_counter() - t0

# --- Polars ---
t0 = time.perf_counter()
for _ in range(50):
    df_pl.group_by("Reporting_Airline").agg([
        pl.col("ArrDelay").mean().alias("mean_arr_delay"),
        pl.col("ArrDelay").std().alias("std_arr_delay"),
    ])
polars_time = time.perf_counter() - t0

print(f"Pandas 50×:  {pandas_time:.3f}s  "
      f"({pandas_time/50*1000:.1f} ms per call)")
print(f"Polars 50×:  {polars_time:.3f}s  "
      f"({polars_time/50*1000:.1f} ms per call)")
print(f"Polars is {pandas_time/polars_time:.1f}× faster")

Rows: 6,847,899
Pandas 50×:  16.408s  (328.2 ms per call)
Polars 50×:  6.174s  (123.5 ms per call)
Polars is 2.7× faster


## Polars vs Pandas

### Loading and inspecting the flights dataset

In [16]:
import polars as pl

cols = ["FlightDate", "Reporting_Airline", "Origin", "Dest",
        "DepDelay", "ArrDelay", "AirTime", "Distance",
        "Cancelled", "CarrierDelay", "WeatherDelay"]

dec_path = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv"
)

df = pl.read_csv(
    dec_path,
    columns=cols,
    infer_schema_length=10000
)

# Schema: column names and their types
print(df.schema)

# Shape
print(df.shape)      # (570394, 11)

# First 5 rows
print(df.head(5))

# Descriptive statistics
print(df.describe())

Schema({'FlightDate': String, 'Reporting_Airline': String, 'Origin': String, 'Dest': String, 'DepDelay': Float64, 'ArrDelay': Float64, 'Cancelled': Float64, 'AirTime': Float64, 'Distance': Float64, 'CarrierDelay': Float64, 'WeatherDelay': Float64})
(570394, 11)
shape: (5, 11)
┌────────────┬───────────────────┬────────┬──────┬───┬─────────┬──────────┬──────────────┬──────────────┐
│ FlightDate ┆ Reporting_Airline ┆ Origin ┆ Dest ┆ … ┆ AirTime ┆ Distance ┆ CarrierDelay ┆ WeatherDelay │
│ ---        ┆ ---               ┆ ---    ┆ ---  ┆   ┆ ---     ┆ ---      ┆ ---          ┆ ---          │
│ str        ┆ str               ┆ str    ┆ str  ┆   ┆ f64     ┆ f64      ┆ f64          ┆ f64          │
╞════════════╪═══════════════════╪════════╪══════╪═══╪═════════╪══════════╪══════════════╪══════════════╡
│ 2023-12-25 ┆ OO                ┆ GRK    ┆ DFW  ┆ … ┆ 30.0    ┆ 134.0    ┆ null         ┆ null         │
│ 2023-12-25 ┆ OO                ┆ LCH    ┆ DFW  ┆ … ┆ 53.0    ┆ 295.0    ┆ null       

In [19]:
# Nice compact view for the book
with pl.Config(tbl_cols=4, tbl_width_chars=100):
    # First 5 rows
    print(df.head(5))

    # Descriptive statistics
    print(df.describe())

shape: (5, 11)
┌────────────┬───────────────────┬───┬──────────────┬──────────────┐
│ FlightDate ┆ Reporting_Airline ┆ … ┆ CarrierDelay ┆ WeatherDelay │
│ ---        ┆ ---               ┆   ┆ ---          ┆ ---          │
│ str        ┆ str               ┆   ┆ f64          ┆ f64          │
╞════════════╪═══════════════════╪═══╪══════════════╪══════════════╡
│ 2023-12-25 ┆ OO                ┆ … ┆ null         ┆ null         │
│ 2023-12-25 ┆ OO                ┆ … ┆ null         ┆ null         │
│ 2023-12-25 ┆ OO                ┆ … ┆ null         ┆ null         │
│ 2023-12-25 ┆ OO                ┆ … ┆ null         ┆ null         │
│ 2023-12-20 ┆ OO                ┆ … ┆ null         ┆ null         │
└────────────┴───────────────────┴───┴──────────────┴──────────────┘
shape: (9, 12)
┌────────────┬────────────┬───┬──────────────┬──────────────┐
│ statistic  ┆ FlightDate ┆ … ┆ CarrierDelay ┆ WeatherDelay │
│ ---        ┆ ---        ┆   ┆ ---          ┆ ---          │
│ str        ┆ str       

### Filtering and column creation

In [20]:
import polars as pl

cols = ["FlightDate", "Reporting_Airline", "Origin", "Dest",
        "DepDelay", "ArrDelay", "AirTime", "Distance", "Cancelled"]

df = pl.read_csv(
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv",
    columns=cols,
    infer_schema_length=10000
)

# Filter: operated (non-cancelled) flights only
operated = df.filter(pl.col("Cancelled") == 0)
print(operated.shape)   # (568084, 9)

# Filter: significantly delayed flights on long routes
long_late = df.filter(
    (pl.col("ArrDelay") > 60) & (pl.col("Distance") > 1000)
)
print(long_late.shape)   # (7944, 9)

# Add new columns
df = df.with_columns([
    # Delay category
    pl.when(pl.col("ArrDelay") <= 0)
      .then(pl.lit("on_time"))
      .when(pl.col("ArrDelay") <= 15)
      .then(pl.lit("minor"))
      .when(pl.col("ArrDelay") <= 60)
      .then(pl.lit("moderate"))
      .otherwise(pl.lit("severe"))
      .alias("delay_category"),

    # Speed in mph (Distance / AirTime * 60)
    (pl.col("Distance") / pl.col("AirTime") * 60)
      .round(1)
      .alias("speed_mph"),

    # Parse FlightDate to actual Date type
    pl.col("FlightDate").str.to_date("%Y-%m-%d").alias("date"),
])

print(df.select(["Reporting_Airline", "ArrDelay",
                 "delay_category", "speed_mph"]).head(4))

(568084, 9)
(7944, 9)
shape: (4, 4)
┌───────────────────┬──────────┬────────────────┬───────────┐
│ Reporting_Airline ┆ ArrDelay ┆ delay_category ┆ speed_mph │
│ ---               ┆ ---      ┆ ---            ┆ ---       │
│ str               ┆ f64      ┆ str            ┆ f64       │
╞═══════════════════╪══════════╪════════════════╪═══════════╡
│ OO                ┆ -19.0    ┆ on_time        ┆ 268.0     │
│ OO                ┆ -18.0    ┆ on_time        ┆ 334.0     │
│ OO                ┆ -24.0    ┆ on_time        ┆ 519.7     │
│ OO                ┆ 8.0      ┆ minor          ┆ 414.3     │
└───────────────────┴──────────┴────────────────┴───────────┘


## Lazy Execution

### What is a LazyFrame?

In [23]:
import polars as pl

dec_path = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv"
)

# Eager: executes immediately, reads full file into memory first
df_eager = (
    pl.read_csv(dec_path, infer_schema_length=10000)
    .filter(pl.col("ArrDelay") > 60)
    .select(["Reporting_Airline", "Origin", "Dest",
             "ArrDelay", "Distance"])
)

# Lazy: builds a logical plan, executes nothing yet
df_lazy = (
    pl.scan_csv(dec_path, infer_schema_length=10000)
    .filter(pl.col("ArrDelay") > 60)
    .select(["Reporting_Airline", "Origin", "Dest",
             "ArrDelay", "Distance"])
)

# Materialise the lazy result
result = df_lazy.collect()
print(result.shape)   # (25116, 5)

(25116, 5)


### Inspecting the query plan

In [24]:
plan = (
    pl.scan_csv(dec_path, infer_schema_length=10000)
    .filter(pl.col("ArrDelay") > 60)
    .select(["Reporting_Airline", "Origin", "Dest",
             "ArrDelay", "Distance"])
)

print(plan.explain())

Csv SCAN [../data/flights/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_12.csv]
PROJECT 5/110 COLUMNS
SELECTION: [(col("ArrDelay")) > (60.0)]
ESTIMATED ROWS: 573407


### Building a lazy pipeline

In [28]:
import polars as pl

dec_path = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv"
)

result = (
    pl.scan_csv(dec_path, infer_schema_length=10000)

    # Keep only the columns we need
    .select(["Reporting_Airline", "Origin", "Dest",
             "DepDelay", "ArrDelay", "Distance", "Cancelled"])

    # Operated flights only
    .filter(pl.col("Cancelled") == 0)

    # Derive a delay severity label
    .with_columns(
        pl.when(pl.col("ArrDelay") <= 0)
          .then(pl.lit("on_time"))
          .when(pl.col("ArrDelay") <= 15)
          .then(pl.lit("minor"))
          .when(pl.col("ArrDelay") <= 60)
          .then(pl.lit("moderate"))
          .otherwise(pl.lit("severe"))
          .alias("delay_category")
    )

    # Compute per-carrier statistics
    .group_by("Reporting_Airline")
    .agg([
        pl.col("ArrDelay").mean().round(2).alias("mean_arr_delay"),
        pl.col("ArrDelay").std().round(2).alias("std_arr_delay"),
        (pl.col("ArrDelay") > 15).sum().alias("delayed_flights"),
        pl.col("ArrDelay").count().alias("total_flights"),
    ])

    # Compute on-time rate and sort
    .with_columns(
        (1 - pl.col("delayed_flights") / pl.col("total_flights"))
          .round(3)
          .alias("on_time_rate")
    )
    .sort("mean_arr_delay")
)
# Nice compact view for the book
with pl.Config(tbl_cols=4, tbl_width_chars=90):
    # Nothing has run yet — call .collect() to execute
    print(result.collect())

shape: (15, 6)
┌───────────────────┬────────────────┬───┬───────────────┬──────────────┐
│ Reporting_Airline ┆ mean_arr_delay ┆ … ┆ total_flights ┆ on_time_rate │
│ ---               ┆ ---            ┆   ┆ ---           ┆ ---          │
│ str               ┆ f64            ┆   ┆ u32           ┆ f64          │
╞═══════════════════╪════════════════╪═══╪═══════════════╪══════════════╡
│ YX                ┆ -9.99          ┆ … ┆ 21449         ┆ 0.919        │
│ DL                ┆ -6.38          ┆ … ┆ 80905         ┆ 0.908        │
│ 9E                ┆ -6.1           ┆ … ┆ 16702         ┆ 0.89         │
│ UA                ┆ -2.2           ┆ … ┆ 59682         ┆ 0.871        │
│ MQ                ┆ -1.13          ┆ … ┆ 20337         ┆ 0.863        │
│ …                 ┆ …              ┆ … ┆ …             ┆ …            │
│ HA                ┆ 4.23           ┆ … ┆ 6597          ┆ 0.856        │
│ G4                ┆ 5.19           ┆ … ┆ 9383          ┆ 0.813        │
│ NK                ┆ 6

### Converting between LazyFrame and DataFrame

In [32]:
# Preview the first 5 rows of a lazy pipeline without full execution
preview = plan.head(5).collect()
print(preview)

shape: (5, 5)
┌───────────────────┬────────┬──────┬──────────┬──────────┐
│ Reporting_Airline ┆ Origin ┆ Dest ┆ ArrDelay ┆ Distance │
│ ---               ┆ ---    ┆ ---  ┆ ---      ┆ ---      │
│ str               ┆ str    ┆ str  ┆ f64      ┆ f64      │
╞═══════════════════╪════════╪══════╪══════════╪══════════╡
│ OO                ┆ LIT    ┆ DEN  ┆ 284.0    ┆ 771.0    │
│ OO                ┆ SUN    ┆ DEN  ┆ 86.0     ┆ 557.0    │
│ OO                ┆ SLC    ┆ PSC  ┆ 619.0    ┆ 521.0    │
│ OO                ┆ SFO    ┆ LAX  ┆ 214.0    ┆ 337.0    │
│ OO                ┆ SFO    ┆ ACV  ┆ 84.0     ┆ 250.0    │
└───────────────────┴────────┴──────┴──────────┴──────────┘


## Fast Aggregations and Joins

### Group-by aggregations

In [34]:
import polars as pl

cols = ["Reporting_Airline", "Origin", "Dest",
        "DepDelay", "ArrDelay", "Distance", "Cancelled"]

df = pl.read_csv(
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv",
    columns=cols,
    infer_schema_length=10000
).filter(pl.col("Cancelled") == 0)

# Per-carrier delay profile for January
carrier_stats = (
    df.group_by("Reporting_Airline")
    .agg([
        pl.col("ArrDelay").count().alias("n_flights"),
        pl.col("ArrDelay").mean().round(2).alias("mean_arr_delay"),
        pl.col("ArrDelay").std().round(2).alias("std_arr_delay"),
        pl.col("ArrDelay").median().round(2).alias("median_arr_delay"),
        pl.col("DepDelay").mean().round(2).alias("mean_dep_delay"),
        pl.col("Distance").mean().round(1).alias("mean_distance"),
    ])
    .sort("mean_arr_delay")
)

# Nice compact view
with pl.Config(tbl_cols=4, tbl_width_chars=90):
    print(carrier_stats)

shape: (15, 7)
┌───────────────────┬───────────┬───┬────────────────┬───────────────┐
│ Reporting_Airline ┆ n_flights ┆ … ┆ mean_dep_delay ┆ mean_distance │
│ ---               ┆ ---       ┆   ┆ ---            ┆ ---           │
│ str               ┆ u32       ┆   ┆ f64            ┆ f64           │
╞═══════════════════╪═══════════╪═══╪════════════════╪═══════════════╡
│ YX                ┆ 21449     ┆ … ┆ -0.07          ┆ 506.1         │
│ DL                ┆ 80905     ┆ … ┆ 4.62           ┆ 965.8         │
│ 9E                ┆ 16702     ┆ … ┆ 5.72           ┆ 440.1         │
│ UA                ┆ 59682     ┆ … ┆ 5.51           ┆ 1136.2        │
│ MQ                ┆ 20337     ┆ … ┆ 4.54           ┆ 546.4         │
│ …                 ┆ …         ┆ … ┆ …              ┆ …             │
│ HA                ┆ 6597      ┆ … ┆ 4.97           ┆ 836.9         │
│ G4                ┆ 9383      ┆ … ┆ 11.45          ┆ 911.2         │
│ NK                ┆ 22461     ┆ … ┆ 15.32          ┆ 990.1  

### Joins

In [35]:
import polars as pl

cols = ["Reporting_Airline", "Origin", "Dest",
        "ArrDelay", "Distance", "Cancelled"]

df = pl.read_csv(
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv",
    columns=cols,
    infer_schema_length=10000
).filter(pl.col("Cancelled") == 0)

# Download the BTS airport lookup table
import urllib.request
airport_url = (
    "https://raw.githubusercontent.com/datasets/"
    "airport-codes/master/data/airport-codes.csv"
)
urllib.request.urlretrieve(airport_url, "../data/flights/airports.csv")

# Load only US airports with IATA codes
airports = (
    pl.read_csv("../data/flights/airports.csv")
    .filter(
        (pl.col("iso_country") == "US") &
        pl.col("iata_code").is_not_null()
    )
    .select([
        pl.col("iata_code").alias("Origin"),
        pl.col("name").alias("origin_name"),
        pl.col("iso_region").str.slice(3).alias("origin_state"),
    ])
)

print(f"Airport lookup rows: {len(airports)}")   # ~1,500

# Inner join: attach origin airport name to each flight
flights_named = df.join(airports, on="Origin", how="left")

print(
    flights_named
    .select(["Reporting_Airline", "Origin", "origin_name",
             "origin_state", "Dest", "ArrDelay"])
    .head(4)
)

Airport lookup rows: 2035
shape: (4, 6)
┌───────────────────┬────────┬─────────────────────────────────┬──────────────┬──────┬──────────┐
│ Reporting_Airline ┆ Origin ┆ origin_name                     ┆ origin_state ┆ Dest ┆ ArrDelay │
│ ---               ┆ ---    ┆ ---                             ┆ ---          ┆ ---  ┆ ---      │
│ str               ┆ str    ┆ str                             ┆ str          ┆ str  ┆ f64      │
╞═══════════════════╪════════╪═════════════════════════════════╪══════════════╪══════╪══════════╡
│ OO                ┆ GRK    ┆ Killeen Regional Airport / Rob… ┆ TX           ┆ DFW  ┆ -19.0    │
│ OO                ┆ LCH    ┆ Lake Charles Regional Airport   ┆ LA           ┆ DFW  ┆ -18.0    │
│ OO                ┆ YUM    ┆ Yuma International Airport / M… ┆ AZ           ┆ DFW  ┆ -24.0    │
│ OO                ┆ ICT    ┆ Wichita Dwight D. Eisenhower N… ┆ KS           ┆ PHX  ┆ 8.0      │
└───────────────────┴────────┴─────────────────────────────────┴──────────────

### Join types

In [36]:
# Anti join: flights whose origin airport is NOT in our lookup table
unmatched = df.join(airports, on="Origin", how="anti")
print(f"Flights with unrecognised origin: {len(unmatched)}")

# Semi join: only keep airports that appear as flight origins
active_origins = airports.join(
    df.select("Origin").unique(), on="Origin", how="semi"
)
print(f"Airports used as origins in January: {len(active_origins)}")

Flights with unrecognised origin: 4160
Airports used as origins in January: 326


### Expressions inside aggregations

In [37]:
import polars as pl

cols = ["Reporting_Airline", "Origin", "ArrDelay",
        "DepDelay", "Distance", "Cancelled"]

df = pl.read_csv(
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv",
    columns=cols,
    infer_schema_length=10000
).filter(pl.col("Cancelled") == 0)

# For each carrier: flights, mean delay, severe delay fraction,
# and fraction of long-haul routes (>1500 miles)
profile = (
    df.group_by("Reporting_Airline")
    .agg([
        pl.col("ArrDelay").count().alias("n_flights"),
        pl.col("ArrDelay").mean().round(2).alias("mean_arr_delay"),
        (pl.col("ArrDelay") > 60).sum().alias("severe_delays"),
        ((pl.col("ArrDelay") > 60).sum()
         / pl.col("ArrDelay").count())
          .round(3).alias("severe_delay_rate"),
        (pl.col("Distance") > 1500).sum().alias("longhaul_flights"),
    ])
    .sort("mean_arr_delay")
)

print(profile.head(6))

shape: (6, 6)
┌───────────────────┬───────────┬────────────────┬───────────────┬───────────────────┬──────────────────┐
│ Reporting_Airline ┆ n_flights ┆ mean_arr_delay ┆ severe_delays ┆ severe_delay_rate ┆ longhaul_flights │
│ ---               ┆ ---       ┆ ---            ┆ ---           ┆ ---               ┆ ---              │
│ str               ┆ u32       ┆ f64            ┆ u32           ┆ f64               ┆ u32              │
╞═══════════════════╪═══════════╪════════════════╪═══════════════╪═══════════════════╪══════════════════╡
│ YX                ┆ 21449     ┆ -9.99          ┆ 520           ┆ 0.024             ┆ 6                │
│ DL                ┆ 80905     ┆ -6.38          ┆ 2163          ┆ 0.027             ┆ 16159            │
│ 9E                ┆ 16702     ┆ -6.1           ┆ 710           ┆ 0.043             ┆ 0                │
│ UA                ┆ 59682     ┆ -2.2           ┆ 2239          ┆ 0.038             ┆ 15132            │
│ MQ                ┆ 20337     

## Processing Large Datasets

### Scanning multiple CSV files at once

In [49]:
import polars as pl

pattern = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_*.csv",
)

# scan_csv reads nothing — just builds a plan over all 12 files
lf = pl.scan_csv(pattern, infer_schema_length=10000)

# Filter, project, and aggregate across the full year lazily
annual_carrier = (
    lf
    .filter(pl.col("Cancelled") == 0)
    .select(["Reporting_Airline", "ArrDelay",
             "DepDelay", "Distance"])
    .group_by("Reporting_Airline")
    .agg([
        pl.col("ArrDelay").count().alias("n_flights"),
        pl.col("ArrDelay").mean().round(2).alias("mean_arr_delay"),
        pl.col("DepDelay").mean().round(2).alias("mean_dep_delay"),
        pl.col("Distance").mean().round(1).alias("mean_distance"),
    ])
    .sort("mean_arr_delay")
    .collect()
)

print(f"Rows: {len(annual_carrier)}")
print(annual_carrier)

Rows: 15
shape: (15, 5)
┌───────────────────┬───────────┬────────────────┬────────────────┬───────────────┐
│ Reporting_Airline ┆ n_flights ┆ mean_arr_delay ┆ mean_dep_delay ┆ mean_distance │
│ ---               ┆ ---       ┆ ---            ┆ ---            ┆ ---           │
│ str               ┆ u32       ┆ f64            ┆ f64            ┆ f64           │
╞═══════════════════╪═══════════╪════════════════╪════════════════╪═══════════════╡
│ YX                ┆ 286489    ┆ -3.38          ┆ 3.16           ┆ 490.1         │
│ 9E                ┆ 196905    ┆ -1.2           ┆ 7.04           ┆ 423.4         │
│ DL                ┆ 972931    ┆ 2.41           ┆ 10.33          ┆ 944.6         │
│ AS                ┆ 242643    ┆ 2.58           ┆ 6.14           ┆ 1402.2        │
│ OH                ┆ 191072    ┆ 3.24           ┆ 7.46           ┆ 424.8         │
│ …                 ┆ …         ┆ …              ┆ …              ┆ …             │
│ AA                ┆ 928058    ┆ 12.3           ┆ 1

### Streaming execution

In [50]:
import polars as pl

pattern = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_*.csv"
)

result = (
    pl.scan_csv(pattern, infer_schema_length=10000)
    .filter(pl.col("Cancelled") == 0)
    .select(["Reporting_Airline", "Origin", "ArrDelay"])
    .group_by(["Reporting_Airline", "Origin"])
    .agg([
        pl.col("ArrDelay").mean().round(2).alias("mean_arr_delay"),
        pl.col("ArrDelay").count().alias("n_flights"),
    ])
    .sort(["Reporting_Airline", "mean_arr_delay"])
    .collect(engine="streaming")   # the new correct way
)

print(result.shape)   # (carrier × origin combinations)
print(result.head(6))

(1729, 4)
shape: (6, 4)
┌───────────────────┬────────┬────────────────┬───────────┐
│ Reporting_Airline ┆ Origin ┆ mean_arr_delay ┆ n_flights │
│ ---               ┆ ---    ┆ ---            ┆ ---       │
│ str               ┆ str    ┆ f64            ┆ u32       │
╞═══════════════════╪════════╪════════════════╪═══════════╡
│ 9E                ┆ LSE    ┆ -18.48         ┆ 151       │
│ 9E                ┆ DFW    ┆ -14.0          ┆ 1         │
│ 9E                ┆ BOS    ┆ -13.73         ┆ 15        │
│ 9E                ┆ OKC    ┆ -13.51         ┆ 55        │
│ 9E                ┆ MOB    ┆ -12.27         ┆ 601       │
│ 9E                ┆ GRB    ┆ -12.26         ┆ 400       │
└───────────────────┴────────┴────────────────┴───────────┘


### Scanning Parquet files

In [51]:
import polars as pl
import glob
import os
import time

cols = ["FlightDate", "Reporting_Airline", "Origin", "Dest",
        "DepDelay", "ArrDelay", "AirTime", "Distance",
        "Cancelled", "CarrierDelay", "WeatherDelay"]

csv_pattern = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_*.csv"
)

# Convert all 12 monthly CSVs to Parquet once
for path in sorted(glob.glob(csv_pattern)):
    out = path.replace(".csv", ".parquet")
    if not os.path.exists(out):
        (
            pl.read_csv(path, columns=cols,
                        infer_schema_length=10000)
            .write_parquet(out, compression="snappy")
        )
        print(f"Converted {os.path.basename(path)}")

# Benchmark: scan CSV vs scan Parquet for the same aggregation
pq_pattern = (
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_*.parquet"
)

t0 = time.perf_counter()
_ = (
    pl.scan_csv(csv_pattern, infer_schema_length=10000)
    .filter(pl.col("Cancelled") == 0)
    .group_by("Reporting_Airline")
    .agg(pl.col("ArrDelay").mean())
    .collect()
)
print(f"scan_csv  (12 months): {time.perf_counter() - t0:.2f}s")

t0 = time.perf_counter()
_ = (
    pl.scan_parquet(pq_pattern)
    .filter(pl.col("Cancelled") == 0)
    .group_by("Reporting_Airline")
    .agg(pl.col("ArrDelay").mean())
    .collect()
)
print(f"scan_parquet (12 months): {time.perf_counter() - t0:.2f}s")

scan_csv  (12 months): 2.28s
scan_parquet (12 months): 0.82s


### Converting to and from Pandas

In [52]:
import polars as pl
import pandas as pd

cols = ["Reporting_Airline", "ArrDelay", "Distance"]

df_polars = pl.read_csv(
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_1.csv",
    columns=cols,
    infer_schema_length=10000
)

# Polars → Pandas
df_pandas = df_polars.to_pandas()
print(type(df_pandas))   # <class 'pandas.core.frame.DataFrame'>
print(df_pandas.shape)   # (538837, 3)

# Pandas → Polars
df_back = pl.from_pandas(df_pandas)
print(type(df_back))     # <class 'polars.dataframe.frame.DataFrame'>

<class 'pandas.DataFrame'>
(538837, 3)
<class 'polars.dataframe.frame.DataFrame'>


## Visualization of Polars DataFrames

In [1]:
import polars as pl
from lets_plot import *
LetsPlot.setup_html()

cols = ["Reporting_Airline", "Origin", "Dest",
        "DepDelay", "ArrDelay", "Distance", "Cancelled"]

df = pl.read_csv(
    "../data/flights/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2023_12.csv",
    columns=cols,
    infer_schema_length=10000
).filter(pl.col("Cancelled") == 0)

# Carrier-level summary used across all four plots
carrier_stats = (
    df.group_by("Reporting_Airline")
    .agg([
        pl.col("ArrDelay").count().alias("n_flights"),
        pl.col("ArrDelay").mean().round(2).alias("mean_arr_delay"),
        pl.col("ArrDelay").std().round(2).alias("std_arr_delay"),
        pl.col("DepDelay").mean().round(2).alias("mean_dep_delay"),
        pl.col("Distance").mean().round(1).alias("mean_distance"),
    ])
    .sort("mean_arr_delay")
)

# --- Plot 1: Bar chart — mean arrival delay by carrier ---
p1 = (
    ggplot(carrier_stats,
           aes(x="Reporting_Airline", y="mean_arr_delay",
               fill="mean_arr_delay")) +
    geom_bar(stat="identity") +
    scale_fill_gradient(low="steelblue", high="tomato") +
    labs(title="Mean Arrival Delay by Carrier",
         x="Carrier", y="Mean Arrival Delay (min)") +
    theme_minimal() +
    theme(legend_position="none")
)

# --- Plot 2: Scatter — dep delay vs arr delay (per carrier) ---
p2 = (
    ggplot(carrier_stats,
           aes(x="mean_dep_delay", y="mean_arr_delay",
               size="n_flights", color="mean_distance")) +
    geom_point(alpha=0.8) +
    geom_text(aes(label="Reporting_Airline"),
              hjust=-0.2, size=7) +
    scale_color_gradient(low="royalblue", high="darkorange") +
    labs(title="Departure vs Arrival Delay",
         x="Mean Departure Delay (min)",
         y="Mean Arrival Delay (min)",
         color="Mean Distance") +
    theme_minimal()
)

# --- Plot 3: Histogram — distribution of per-flight arrival delays ---
operated = df.filter(pl.col("ArrDelay").is_not_null())

p3 = (
    ggplot(operated, aes(x="ArrDelay")) +
    geom_histogram(bins=60, fill="steelblue", color="white",
                   alpha=0.8) +
    coord_cartesian(xlim=[-60, 180]) +
    labs(title="Distribution of Arrival Delays (Dec 2023)",
         x="Arrival Delay (min)", y="Flight Count") +
    theme_minimal()
)

# --- Plot 4: Box plot — arrival delay spread by carrier ---
p4 = (
    ggplot(
        df.filter(
            pl.col("ArrDelay").is_not_null() &
            pl.col("Reporting_Airline").is_in(
                ["DL", "UA", "AA", "WN", "B6", "NK", "F9", "AS"]
            )
        ),
        aes(x="Reporting_Airline", y="ArrDelay",
            fill="Reporting_Airline")
    ) +
    geom_boxplot(outlier_alpha=0.2) +
    coord_cartesian(ylim=[-40, 120]) +
    labs(title="Arrival Delay Spread — Selected Carriers",
         x="Carrier", y="Arrival Delay (min)") +
    theme_minimal() +
    theme(legend_position="none")
)

# Combine into a 2×2 grid
grid = gggrid([p1, p2, p3, p4], ncol=2)

# Combine into a 2×2 grid
grid = gggrid([p1, p2, p3, p4], ncol=2)

# Save the plot
from pathlib import Path
output_path = Path("../outputs/").resolve()

# Save as interactive HTML
plot_html = "polars_ggplot_flight_performance.html"
ggsave(grid, filename=str(output_path / plot_html))

# Save as PNG (scaled)
plot_png = "polars_ggplot_flight_performance.png"
ggsave(grid, filename=str(output_path / plot_png), scale=2)

grid.show()